# SLM Training Run

Launch and monitor any training stage directly from the notebook.
Useful for interactive experimentation — tweak hyperparameters, inspect
checkpoints, resume from interruption, all without leaving the notebook.

**Stages:** pretrain → sft-chat → sft-code → dpo

In [ ]:
import subprocess
import os
import time
import json
from pathlib import Path
import torch

REPO_ROOT   = Path("/workspace/slm")
DATA_DIR    = Path("/data")
RESULTS_DIR = Path("/results")

print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory // 1024**3}GB")

## Environment Check

Verify all prerequisites before starting a training run.

In [ ]:
def check_prerequisites(stage: str):
    checks = {
        "pretrain": [
            (DATA_DIR / "curated/tokenized/text_document.bin", "Tokenized dataset"),
            (DATA_DIR / "tokenizer/slm_tokenizer.model",      "Tokenizer model"),
        ],
        "sft-chat": [
            (DATA_DIR / "sft/chat/train.jsonl",                "Chat SFT train data"),
            (RESULTS_DIR / "slm_gpt_125m/checkpoints",         "Pretrain checkpoint dir"),
        ],
        "sft-code": [
            (DATA_DIR / "sft/code/train.jsonl",                "Code SFT train data"),
            (RESULTS_DIR / "slm_sft_chat/checkpoints",         "Chat SFT checkpoint dir"),
        ],
        "dpo": [
            (DATA_DIR / "dpo/train.jsonl",                     "DPO train data"),
            (RESULTS_DIR / "slm_sft_code/checkpoints",         "Code SFT checkpoint dir"),
        ],
    }

    print(f"Prerequisites for stage: {stage}")
    all_ok = True
    for path, label in checks.get(stage, []):
        exists = Path(path).exists()
        status = "✓" if exists else "✗ MISSING"
        print(f"  {status}  {label}: {path}")
        if not exists:
            all_ok = False
    return all_ok

check_prerequisites("pretrain")

## Pre-Training

In [ ]:
# Configuration
PRETRAIN_CONFIG = REPO_ROOT / "pretrain/configs/gpt_125m.yaml"
PRETRAIN_GPUS   = torch.cuda.device_count()
PRETRAIN_WANDB  = False  # set True to enable W&B logging

assert check_prerequisites("pretrain"), "Fix missing prerequisites before continuing"
print(f"\nReady to launch pretrain with {PRETRAIN_GPUS} GPU(s)")
print(f"Config: {PRETRAIN_CONFIG}")

In [ ]:
# Launch pre-training
# Streams output to the notebook cell
cmd = [
    "bash", str(REPO_ROOT / "pretrain/scripts/train.sh"),
    "--config", str(PRETRAIN_CONFIG),
    "--gpus", str(PRETRAIN_GPUS),
]
if PRETRAIN_WANDB:
    cmd.append("--wandb")

print("Launching:", " ".join(cmd))
print("Output streaming...\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

## SFT — Chat Stage

In [ ]:
assert check_prerequisites("sft-chat"), "Fix missing prerequisites before continuing"

cmd = [
    "bash", str(REPO_ROOT / "finetune/scripts/train_sft.sh"),
    "--stage", "chat",
    "--gpus", str(torch.cuda.device_count()),
]

print("Launching SFT chat stage...")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

## SFT — Code Stage

In [ ]:
assert check_prerequisites("sft-code"), "Fix missing prerequisites before continuing"

cmd = [
    "bash", str(REPO_ROOT / "finetune/scripts/train_sft.sh"),
    "--stage", "code",
    "--gpus", str(torch.cuda.device_count()),
]

print("Launching SFT code stage...")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

## DPO Alignment

In [ ]:
DPO_BETA = 0.1  # adjust if win rate is low (try 0.05) or capabilities degrade (try 0.2)

assert check_prerequisites("dpo"), "Fix missing prerequisites before continuing"

cmd = [
    "bash", str(REPO_ROOT / "alignment/scripts/train_dpo.sh"),
    "--gpus", str(torch.cuda.device_count()),
    "--beta", str(DPO_BETA),
]

print(f"Launching DPO alignment (beta={DPO_BETA})...")
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print(f"\nProcess exited with code {proc.returncode}")

## Checkpoint Browser

List all available checkpoints across all stages.

In [ ]:
stages = [
    ("Pretrain",  RESULTS_DIR / "slm_gpt_125m/checkpoints"),
    ("SFT chat",  RESULTS_DIR / "slm_sft_chat/checkpoints"),
    ("SFT code",  RESULTS_DIR / "slm_sft_code/checkpoints"),
    ("DPO",       RESULTS_DIR / "slm_dpo/checkpoints"),
]

for stage_name, ckpt_dir in stages:
    checkpoints = sorted(Path(ckpt_dir).glob("*.nemo")) if Path(ckpt_dir).exists() else []
    print(f"\n{stage_name} ({len(checkpoints)} checkpoints):")
    for ckpt in checkpoints:
        size_gb = ckpt.stat().st_size / 1e9
        print(f"  {ckpt.name:<60} {size_gb:.1f}GB")
    if not checkpoints:
        print("  (none yet)")